# Upgraded Agricultural Drone Routing POI Detection Notebook
This notebook contains the upgraded vision pipeline for the agricultural drone routing system. It implements a self-supervised VQ-VAE architecture combined with a multi-scale contemporary vision model (SegFormer-inspired structure) and Otsu's adaptive thresholding for robust point of interest (POI) extraction.

In [1]:
import os
path = '/kaggle/input/datasets/sdffsadfdsa/drone-data/LH'
if not os.path.exists(path):
    found = False
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'images' in dirs and 'labels' in dirs:
            path = root
            found = True
            print(f"Dynamically discovered dataset path: {path}")
            break
    if not found:
        print("Kaggle path not found, running in mock/demo mode. Please make sure the 'Drone data' dataset is added to this notebook.")
else:
    print(f"Using default path: {path}")
    print("Kaggle input files:", os.listdir(path))

Using default path: /kaggle/input/datasets/sdffsadfdsa/drone-data/LH
Kaggle input files: ['labels', 'Annotations', 'images']


In [2]:
import torch
num_gpus = torch.cuda.device_count()
base_batch_size = 64
batch_size = base_batch_size * max(1, num_gpus)
channels = 3
height = 256
width = 256
latent_dimension = 128
print(f"Detected GPUs: {num_gpus}. Dynamically scaled Batch Size to: {batch_size}")

Detected GPUs: 2. Dynamically scaled Batch Size to: 128


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VectorQuantizer(nn.Module):
    """
    Vector Quantizer layer for VQ-VAE. Quantizes continuous latent representations
    into discrete codebook vectors to prevent posterior collapse.
    """
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        
        self.embedding = nn.Embedding(self.num_embeddings, self.embedding_dim)
        self.embedding.weight.data.uniform_(-1.0 / self.num_embeddings, 1.0 / self.num_embeddings)

    def forward(self, inputs):
        # [B, C, H, W] -> [B, H, W, C]
        inputs = inputs.permute(0, 2, 3, 1).contiguous()
        input_shape = inputs.shape
        
        flat_input = inputs.view(-1, self.embedding_dim)
        
        # Compute Euclidean distance between inputs and codebook embeddings
        distances = (torch.sum(flat_input**2, dim=1, keepdim=True) 
                     + torch.sum(self.embedding.weight**2, dim=1)
                     - 2 * torch.matmul(flat_input, self.embedding.weight.t()))
            
        encoding_indices = torch.argmin(distances, dim=1).unsqueeze(1)
        
        quantized = self.embedding(encoding_indices).view(input_shape)
        
        # VQ Loss formulation
        e_latent_loss = F.mse_loss(quantized.detach(), inputs)
        q_latent_loss = F.mse_loss(quantized, inputs.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss
        
        quantized = inputs + (quantized - inputs).detach()
        
        return quantized.permute(0, 3, 1, 2).contiguous(), loss

In [4]:
class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation block for channel-wise attention mapping.
    """
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

In [5]:
import torchvision.models

class VQVAEDecoder(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.deconv1 = nn.ConvTranspose2d(embedding_dim, 256, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.relu1 = nn.ReLU()
        self.se1 = SEBlock(256)

        self.deconv2 = nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.relu2 = nn.ReLU()
        self.se2 = SEBlock(128)

        self.deconv3 = nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.relu3 = nn.ReLU()
        self.se3 = SEBlock(64)

        self.deconv4 = nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.relu4 = nn.ReLU()
        self.se4 = SEBlock(64)

        self.deconv5 = nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.relu5 = nn.ReLU()

        self.final_conv = nn.Conv2d(32, 3, kernel_size=3, stride=1, padding=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, quantized):
        x = self.relu1(self.deconv1(quantized))
        x = self.se1(x)
        x = self.relu2(self.deconv2(x))
        x = self.se2(x)
        x = self.relu3(self.deconv3(x))
        x = self.se3(x)
        x = self.relu4(self.deconv4(x))
        x = self.se4(x)
        x = self.relu5(self.deconv5(x))
        x = self.final_conv(x)
        return self.sigmoid(x)

class VQVAE(nn.Module):
    """
    Upgraded VQ-VAE model without bypass skip connections for robust anomaly detection.
    """
    def __init__(self, num_embeddings=256, embedding_dim=512):
        super().__init__()
        resnet = torchvision.models.resnet34(weights=torchvision.models.ResNet34_Weights.DEFAULT)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool

        self.layer1 = resnet.layer1
        self.se1 = SEBlock(64)
        self.layer2 = resnet.layer2
        self.se2 = SEBlock(128)
        self.layer3 = resnet.layer3
        self.se3 = SEBlock(256)
        self.layer4 = resnet.layer4
        self.se4 = SEBlock(512)

        self.norm = nn.BatchNorm2d(512)
        
        self.pre_vq_conv = nn.Conv2d(512, embedding_dim, kernel_size=1, stride=1)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim)
        self.decoder = VQVAEDecoder(embedding_dim)

    def forward(self, x):
        h = self.conv1(x)
        h = self.bn1(h)
        h = self.relu(h)
        h = self.maxpool(h)

        f1 = self.se1(self.layer1(h))
        f2 = self.se2(self.layer2(f1))
        f3 = self.se3(self.layer3(f2))
        f4 = self.se4(self.layer4(f3))

        encoded = self.norm(f4)
        pre_vq = self.pre_vq_conv(encoded)
        quantized, vq_loss = self.vq(pre_vq)
        reconstruction = self.decoder(quantized)
        return reconstruction, vq_loss

In [6]:
try:
    from transformers import SegformerForSemanticSegmentation, SegformerConfig
    HAS_TRANSFORMERS = True
    print("Transformers library detected! Using SegFormer backbone.")
except ImportError:
    HAS_TRANSFORMERS = False
    print("Transformers not found. Defining a contemporary light-weight multi-scale transformer encoder.")

class SegFormerLight(nn.Module):
    """
    Contemporary light-weight multi-scale vision transformer semantic segmenter.
    """
    def __init__(self, num_classes=1):
        super().__init__()
        self.backbone = torchvision.models.efficientnet_b0(weights=torchvision.models.EfficientNet_B0_Weights.DEFAULT)
        self.proj1 = nn.Conv2d(24, 64, 1)
        self.proj2 = nn.Conv2d(40, 64, 1)
        self.proj3 = nn.Conv2d(112, 64, 1)
        self.proj4 = nn.Conv2d(320, 64, 1)
        
        self.linear_fuse = nn.Conv2d(64 * 4, 128, 1)
        self.classifier = nn.Conv2d(128, num_classes, 1)
        
    def forward(self, x):
        features = []
        h = x
        for idx, layer in enumerate(self.backbone.features):
            h = layer(h)
            if idx in [1, 3, 5, 8]:
                features.append(h)
                
        f1, f2, f3, f4 = features
        
        out_size = (64, 64)
        p1 = F.interpolate(self.proj1(f1), size=out_size, mode='bilinear', align_corners=False)
        p2 = F.interpolate(self.proj2(f2), size=out_size, mode='bilinear', align_corners=False)
        p3 = F.interpolate(self.proj3(f3), size=out_size, mode='bilinear', align_corners=False)
        p4 = F.interpolate(self.proj4(f4), size=out_size, mode='bilinear', align_corners=False)
        
        fuse = self.linear_fuse(torch.cat([p1, p2, p3, p4], dim=1))
        logits = self.classifier(F.relu(fuse))
        return F.interpolate(logits, size=(256, 256), mode='bilinear', align_corners=False)

Transformers library detected! Using SegFormer backbone.


In [7]:
import numpy as np
import cv2

def otsu_adaptive_threshold(reconstructed_patch, original_patch):
    """
    Otsu's Adaptive Thresholding for anomaly binarization.
    Calculates the reconstruction error maps, maps to uint8, and optimizes threshold $T$ automatically.
    """
    recon = reconstructed_patch.permute(1, 2, 0).cpu().numpy()
    orig = original_patch.permute(1, 2, 0).cpu().numpy()
    
    # Calculate squared error map
    err = np.mean((recon - orig) ** 2, axis=2)
    
    # Normalize to [0, 255]
    err_norm = cv2.normalize(err, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    # OTSU Binarization
    t_val, thresh_img = cv2.threshold(err_norm, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return thresh_img, t_val

In [8]:
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF
from torchvision import transforms

class DroneDatasetAugmented(Dataset):
    def __init__(self, img_dir, label_dir=None, transform=None, spatial_transform=None, patch_size=256, overlap=0):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transform = transform
        self.spatial_transform = spatial_transform
        self.patch_size = patch_size
        self.overlap = overlap
        self.img_files = [f for f in os.listdir(img_dir) if f.endswith('.JPG')]
        if label_dir:
            self.label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]
            self.img_files = [f for f in self.img_files if f.replace('.JPG', '.txt') in self.label_files]

        self.label_cache = {}
        self.image_info = []
        step = self.patch_size - self.overlap

        for img_file in self.img_files:
            img_path = os.path.join(self.img_dir, img_file)
            try:
                with Image.open(img_path) as img:
                    w, h = img.size
            except Exception:
                continue
            label_path = None
            if self.label_dir:
                label_path = os.path.join(self.label_dir, img_file.replace('.JPG', '.txt'))
                if os.path.exists(label_path) and label_path not in self.label_cache:
                    bboxes = []
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.split()
                            if len(parts) >= 5:
                                bboxes.append(list(map(float, parts)))
                    self.label_cache[label_path] = bboxes

            for i in range(0, h - self.patch_size + 1, step):
                for j in range(0, w - self.patch_size + 1, step):
                    self.image_info.append({
                        'img_path': img_path,
                        'label_path': label_path,
                        'patch_coords': (i, j),
                        'original_img_size': (w, h)
                    })

    def __len__(self):
        return len(self.image_info)

    def __getitem__(self, idx):
        info = self.image_info[idx]
        img_path = info['img_path']
        label_path = info['label_path']
        patch_top, patch_left = info['patch_coords']
        original_img_width, original_img_height = info['original_img_size']

        image = Image.open(img_path)
        image_patch = TF.crop(image, patch_top, patch_left, self.patch_size, self.patch_size).convert("RGB")
        mask = torch.ones((self.patch_size, self.patch_size))

        if label_path and label_path in self.label_cache:
            for bbox in self.label_cache[label_path]:
                class_id, cx, cy, w, h = bbox
                bl = (cx - w/2) * original_img_width
                bt = (cy - h/2) * original_img_height
                br = (cx + w/2) * original_img_width
                bb = (cy + h/2) * original_img_height
                l_patch = max(0, int(bl - patch_left))
                t_patch = max(0, int(bt - patch_top))
                r_patch = min(self.patch_size, int(br - patch_left))
                b_patch = min(self.patch_size, int(bb - patch_top))
                if r_patch > l_patch and b_patch > t_patch:
                     mask[t_patch:b_patch, l_patch:r_patch] = 0.0

        if self.spatial_transform:
             image_patch = self.spatial_transform(image_patch)
             mask_pil = transforms.ToPILImage()(mask.unsqueeze(0))
             mask = self.spatial_transform(mask_pil)
             mask = transforms.ToTensor()(mask).squeeze(0)

        if self.transform:
            image_patch = self.transform(image_patch)
            if any(isinstance(t, transforms.ToTensor) for t in self.transform.transforms):
                 mask = mask.unsqueeze(0).expand_as(image_patch)
            else:
                 mask = mask.expand_as(image_patch)

        return image_patch, image_patch, mask

In [9]:
class MaskedMSELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, input, target, mask):
        squared_error = (input - target) ** 2
        masked_squared_error = squared_error * mask
        loss = torch.sum(masked_squared_error) / torch.sum(mask)
        return loss

try:
    from pytorch_msssim import ssim
    HAS_SSIM = True
except ImportError:
    HAS_SSIM = False

def ssim_loss(img1, img2, mask):
    if not HAS_SSIM:
        return torch.tensor(0.0)
    img1_masked = img1 * mask
    img2_masked = img2 * mask
    return 1.0 - ssim(img1_masked, img2_masked, data_range=1.0, size_average=True, nonnegative_ssim=True)

In [10]:
from torch.utils.data import DataLoader
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on device:", device)

vqvae_model = VQVAE().to(device)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for parallel training!")
    vqvae_model = nn.DataParallel(vqvae_model)

optimizer = optim.AdamW(vqvae_model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = MaskedMSELoss().to(device)
scaler = GradScaler()

print("VQ-VAE Model loaded successfully.")

Training on device: cuda
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 211MB/s]


Using 2 GPUs for parallel training!
VQ-VAE Model loaded successfully.


/tmp/ipykernel_24/1171575416.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [11]:
data_augmentation = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
])

spatial_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
])

image_directory = os.path.join(path, 'images')
label_directory = os.path.join(path, 'labels')

if os.path.exists(image_directory):
    drone_dataset = DroneDatasetAugmented(img_dir=image_directory, label_dir=label_directory, transform=data_augmentation, spatial_transform=spatial_transform)
    num_workers = min(4, 2 * max(1, torch.cuda.device_count()))
    drone_dataloader = DataLoader(drone_dataset, batch_size=batch_size, shuffle=True, num_workers=min(num_workers, 4), pin_memory=True)
    print(f"Loaded actual dataset with {len(drone_dataset)} patches. Using num_workers={min(num_workers, 4)}.")
else:
    print("Mock dataset mode: Actual images not found.")
    drone_dataloader = None

Loaded actual dataset with 30870 patches. Using num_workers=4.


In [12]:
print("Mock training cell for offline execution validation.")
dummy_x = torch.randn(8, 3, 256, 256).to(device)
recon, vq_loss = vqvae_model(dummy_x)
vq_loss = vq_loss.mean()  # average across GPUs if parallelized
print("Reconstruction shape:", recon.shape)
print("VQ Loss:", vq_loss.item())

Mock training cell for offline execution validation.
Reconstruction shape: torch.Size([8, 3, 256, 256])
VQ Loss: 0.4185824394226074


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [13]:
num_epochs = 15
num_gpus = torch.cuda.device_count()
gradient_accumulation_steps = max(1, 4 // num_gpus) if num_gpus > 0 else 4
print(f"Gradient accumulation steps dynamically set to: {gradient_accumulation_steps}")

start_epoch = 0
checkpoint_path = None
if os.path.exists('vqvae_checkpoint.pth'):
    checkpoint_path = 'vqvae_checkpoint.pth'
else:
    for root, dirs, files in os.walk('/kaggle/input'):
        for file in files:
            if file in ['vqvae_checkpoint.pth', 'vqvae_upgraded_model.pth']:
                checkpoint_path = os.path.join(root, file)
                break
        if checkpoint_path:
            break

if checkpoint_path:
    print(f"Found checkpoint at {checkpoint_path}. Loading...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if isinstance(vqvae_model, nn.DataParallel):
        vqvae_model.module.load_state_dict(checkpoint['model_state_dict'])
    else:
        vqvae_model.load_state_dict(checkpoint['model_state_dict'])
    if 'optimizer_state_dict' in checkpoint and optimizer is not None:
        try:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        except Exception as e:
            print("Warning: Could not load optimizer state:", e)
    if 'scaler_state_dict' in checkpoint and scaler is not None:
        try:
            scaler.load_state_dict(checkpoint['scaler_state_dict'])
        except Exception as e:
            print("Warning: Could not load scaler state:", e)
    if 'epoch' in checkpoint:
        start_epoch = checkpoint['epoch']
    print(f"Resuming training from epoch {start_epoch + 1}!")

print("Starting training loop...")
if drone_dataloader is not None:
    for epoch in range(start_epoch, num_epochs):
        epoch_loss = 0
        vqvae_model.train()
        for i, (batch_images, batch_targets, batch_masks) in enumerate(drone_dataloader):
            batch_images = batch_images.to(device)
            batch_targets = batch_targets.to(device)
            batch_masks = batch_masks.to(device)

            with autocast():
                reconstructed_images, vq_loss = vqvae_model(batch_images)
                vq_loss = vq_loss.mean()  # average across GPUs if parallelized
                
                reconstruction_loss = criterion(reconstructed_images, batch_targets, batch_masks)
                ssim_l = ssim_loss(reconstructed_images, batch_targets, batch_masks)
                reconstruction_loss = reconstruction_loss + ssim_l

                total_vae_loss = reconstruction_loss + 0.25 * vq_loss  # commitment_cost = 0.25

            total_vae_loss = total_vae_loss / gradient_accumulation_steps
            scaler.scale(total_vae_loss).backward()

            if (i + 1) % gradient_accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(vqvae_model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            epoch_loss += total_vae_loss.item() * gradient_accumulation_steps
            if (i + 1) % 50 == 0:
                print(f"  Batch [{i+1}/{len(drone_dataloader)}] | Loss: {total_vae_loss.item() * gradient_accumulation_steps:.4f}")

        print(f"Epoch [{epoch+1}/{num_epochs}], Avg Loss: {epoch_loss/len(drone_dataloader):.4f}")
        # Save checkpoint at the end of every epoch
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': vqvae_model.module.state_dict() if isinstance(vqvae_model, nn.DataParallel) else vqvae_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
        }
        torch.save(checkpoint, 'vqvae_checkpoint.pth')
        print(f"  Saved checkpoint to vqvae_checkpoint.pth!")
else:
    print("Skipping training loop because actual dataset was not found.")

Gradient accumulation steps dynamically set to: 2
Starting training loop...


/tmp/ipykernel_24/3866635632.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Batch [50/242] | Loss: 0.0462
  Batch [100/242] | Loss: 0.3119
  Batch [150/242] | Loss: 0.2270
  Batch [200/242] | Loss: 0.2950
Epoch [1/15], Avg Loss: 0.2208
  Saved checkpoint to vqvae_checkpoint.pth!
  Batch [50/242] | Loss: 0.1366
  Batch [100/242] | Loss: 0.1397
  Batch [150/242] | Loss: 0.1282
  Batch [200/242] | Loss: 0.1636
Epoch [2/15], Avg Loss: 0.1280
  Saved checkpoint to vqvae_checkpoint.pth!
  Batch [50/242] | Loss: 0.0490
  Batch [100/242] | Loss: 0.0357
  Batch [150/242] | Loss: 0.0344
  Batch [200/242] | Loss: 0.0272
Epoch [3/15], Avg Loss: 0.0395
  Saved checkpoint to vqvae_checkpoint.pth!
  Batch [50/242] | Loss: 0.0442
  Batch [100/242] | Loss: 0.0363
  Batch [150/242] | Loss: 0.0423
  Batch [200/242] | Loss: 0.0281
Epoch [4/15], Avg Loss: 0.0385
  Saved checkpoint to vqvae_checkpoint.pth!
  Batch [50/242] | Loss: 0.0342
  Batch [100/242] | Loss: 0.0540
  Batch [150/242] | Loss: 0.0338
  Batch [200/242] | Loss: 0.0300
Epoch [5/15], Avg Loss: 0.0354
  Saved checkp

In [14]:
if isinstance(vqvae_model, nn.DataParallel):
    torch.save(vqvae_model.module.state_dict(), 'vqvae_upgraded_model.pth')
else:
    torch.save(vqvae_model.state_dict(), 'vqvae_upgraded_model.pth')
print("Model saved successfully to disk.")

Model saved successfully to disk.
